# Navigating hierarchy in Odeon

In Odeon, objects are stored in a tree data structure. That means each object can have a parent object and multiple children objects.

Per object type, allowed children types are defined. You can find the allowed types in the code (but not in the documentation yet). Below is an example from the `Building` class:

```python
class Building:
    ...
    # children attributes:
    _CHILDREN_ATTRIBUTES = {
        "_building_thermal_zone": "BuildingThermalZone",
        "_building_geometry_nominal": "FootprintNominalBuildingGeometry",
        "_building_geometry_cuboid": "RoofedCuboidBuildingGeometry",
        "_building_elements": "BuildingElement[]",
        "_building_units": "BuildingUnit[]",
    }
```

It shows that a `Building` can have children of types `BuildingThermalZone`, `FootprintNominalBuildingGeometry`, `RoofedCuboidBuildingGeometry`, `BuildingElement`, and `BuildingUnit`.

## Children and Parent

Parent-child relationships are established when adding an object to another object. For example, when you add a BuildingUnit to a Building, the Building becomes the parent of the BuildingUnit, and the BuildingUnit becomes a child of the Building.

Functions for adding objects are typically named `add_<object_type>s`, e.g. `add_building_units`. You can also use `add_children`. This will try to store the objects in the appropriate children attributes based on their type.

In [10]:
from odeon.model import Building, BuildingUnit

some_building = Building(name="Building A")
building_unit_1 = BuildingUnit(name="Unit 1")

print("building_unit_1.parent before:", building_unit_1.parent)  # None before adding to building

# add unit to building:
some_building.add_building_units(building_unit_1)

print("building_unit_1.parent after:", building_unit_1.parent)  # Should now reference building_a

# the children of building_a should include building_unit_1
print("building_a.children:", some_building.children)  # Should include building_unit_1

building_unit_1.parent before: None
building_unit_1.parent after: Building(id=16, name='Building A')
building_a.children: [FootprintNominalBuildingGeometry(id=15), EnergySystem(id=17), BuildingUnit(id=18, name='Unit 1')]


As you can see, the Building has the BuildingUnit as a child, and the BuildingUnit has the Building as its parent. The building has also two additional children. These sub-objects are created automatically when creating a Building.

When you are interested only in children of a specific type, you usually use the property named `<object_type>s`, e.g. `building_units`:

In [2]:
some_building.building_units

[BuildingUnit(id=3, name='Unit 1')]

## Navigating more than one level

We can also build more complex hierarchies by adding objects to other objects. For example, we can add Residents to a Household (a Household is a subtype of BuildingUnit), and then add the Household to a Building:

In [3]:
from odeon.model import Building, Household, Commercial, Resident

mixed_building = Building(name="Building B")
household_1 = Household(name="Household 1")
commercial_1 = Commercial(name="Commercial 1")

mixed_building.add_building_units([household_1, commercial_1])

resident_1 = Resident(name="Resident 1")
resident_2 = Resident(name="Resident 2")
household_1.add_residents([resident_1, resident_2])

print("household.children:", household_1.children)
print("resident_1.parent:", resident_1.parent)

household.children: [EnergySystem(id=9), Resident(id=12, name='Resident 1'), Resident(id=13, name='Resident 2')]
resident_1.parent: Household(id=8, name='Household 1')


In addition to accessing direct parents and children, Odeon provides functions to navigate the entire hierarchy.
For example, you can use the properties `offspring`, `ancestors`, and `root` to access all descendants, all ancestors, and the root object of any object in the hierarchy:

In [4]:
print("building_b.offspring:")
for obj in mixed_building.offspring:
    print(" -", obj)

print("\nresident_1.ancestors:")
for ancestor in resident_1.ancestors:
    print(" -", ancestor)

print("\nresident_1.root:", resident_1.root)

building_b.offspring:
 - FootprintNominalBuildingGeometry(id=5)
 - EnergySystem(id=7)
 - Household(id=8, name='Household 1')
 - EnergySystem(id=9)
 - Commercial(id=10, name='Commercial 1')
 - EnergySystem(id=11)
 - Resident(id=12, name='Resident 1')
 - Resident(id=13, name='Resident 2')

resident_1.ancestors:
 - Household(id=8, name='Household 1')
 - Building(id=6, name='Building B')

resident_1.root: Building(id=6, name='Building B')


To visualize we print the entire tree structure of an object and its descendants:

In [5]:
from odeon.processing.prints import print_object_tree

print_object_tree(mixed_building)

└─Building(id=6, name='Building B')
  ├─FootprintNominalBuildingGeometry(id=5)
  ├─EnergySystem(id=7)
  ├─Household(id=8, name='Household 1')
  │ ├─EnergySystem(id=9)
  │ ├─Resident(id=12, name='Resident 1')
  │ └─Resident(id=13, name='Resident 2')
  └─Commercial(id=10, name='Commercial 1')
    └─EnergySystem(id=11)


## Find objects

You can find offspring objects of a specific type, name, or ID:

In [6]:
print("Households:", mixed_building.find_objects(type=Household))
print("named 'Commercial 1':", mixed_building.find_objects(name="Commercial 1"))
print("ID=2:", mixed_building.find_objects(id=resident_2.id))

# when you are looking for a single object, you can use `find_object`:
print("named 'Household 1':", mixed_building.find_object(name="Resident 1"))

Households: [Household(id=8, name='Household 1')]
named 'Commercial 1': [Commercial(id=10, name='Commercial 1')]
ID=2: [Resident(id=13, name='Resident 2')]
named 'Household 1': Resident(id=12, name='Resident 1')


You can also search objects using filters:

In [7]:
from odeon.model import EnergySystem

# each building and each building unit has an energy system upon creation, so we expect to find three here:
x = mixed_building.find_objects(EnergySystem)
print("Energy systems in mixed_building:", x)

# using filters:
x = mixed_building.find_objects_filtered(type=EnergySystem, only_reachable_through=Household)
print("Energy systems in mixed_building reachable through Household:", x)

# this is obviously equivalent to:
x = household_1.find_objects(type=EnergySystem)
print("Energy systems in household_1:", x)

# using types as blacklist:
x = mixed_building.find_objects_filtered(type=EnergySystem, omit_reachable_through=Household)
print("Objects in mixed_building not reachable through Household:", x)

Energy systems in mixed_building: [EnergySystem(id=7), EnergySystem(id=9), EnergySystem(id=11)]
Energy systems in mixed_building reachable through Household: [EnergySystem(id=9)]
Energy systems in household_1: [EnergySystem(id=9)]
Objects in mixed_building not reachable through Household: [EnergySystem(id=7), EnergySystem(id=11)]


## Branches as hierarchy elements

You can use branches to store any number of objects. These objects will be the children of the branch, and the branch will be the parent of these objects. Many functions that work with other hierarchy elements also work with branches:

In [8]:
from odeon.model import Branch

branch = Branch(name="My Branch", year=2024)
branch.add_objects([some_building, mixed_building])

print("branch.children:", branch.children)
print("branch.offspring:", branch.offspring)

branch.children: [Building(id=1, name='Building A'), Building(id=6, name='Building B')]
branch.offspring: [FootprintNominalBuildingGeometry(id=0), Building(id=1, name='Building A'), EnergySystem(id=2), BuildingUnit(id=3, name='Unit 1'), EnergySystem(id=4), FootprintNominalBuildingGeometry(id=5), Building(id=6, name='Building B'), EnergySystem(id=7), Household(id=8, name='Household 1'), EnergySystem(id=9), Commercial(id=10, name='Commercial 1'), EnergySystem(id=11), Resident(id=12, name='Resident 1'), Resident(id=13, name='Resident 2')]


Note that the branch won't get listed as the root. It can be accessed via the `branch` property of the child objects, though:

In [9]:
print("household_1.root:", household_1.root)
print("household_1.branch:", household_1.branch)

household_1.root: Building(id=6, name='Building B')
household_1.branch: Branch(id=14, n_objects=2, year=2024, name='My Branch')
